In [1]:
import pandas as pd
from loguru import logger
import os

In [2]:
COLS_TO_DROP = [
    # --- Negligible numerical ---
    "mths_since_last_delinq",
    "mths_since_last_record",
    "total_acc",
    "collections_12_mths_ex_med",
    "mths_since_last_major_derog",
    "acc_now_delinq",
    "tot_coll_amt",
    "mths_since_rcnt_il",
    "total_bal_il",
    "max_bal_bc",
    "total_cu_tl",
    "mo_sin_old_il_acct",
    "num_il_tl",
    "pct_tl_nvr_dlq",
    # --- Unanalysed / low-info numerical ---
    "chargeoff_within_12_mths",
    "delinq_amnt",
    "mo_sin_rcnt_rev_tl_op",
    "mo_sin_rcnt_tl",
    "mths_since_recent_bc",
    "mths_since_recent_inq",
    "mths_since_recent_revol_delinq",
    "num_actv_bc_tl",
    "num_bc_sats",
    "num_bc_tl",
    "num_sats",
    "num_tl_120dpd_2m",
    "num_tl_30dpd",
    "num_tl_90g_dpd_24m",
    "num_tl_op_past_12m",
    "tax_liens",
    "total_bal_ex_mort",
    "total_il_high_credit_limit",
    "bc_open_to_buy",
    # --- Joint application columns ---
    "application_type",           # after filtering Individual
    "annual_inc_joint",
    "dti_joint",
    "verification_status_joint",
    "revol_bal_joint",
    "sec_app_earliest_cr_line",
    "sec_app_inq_last_6mths",
    "sec_app_mort_acc",
    "sec_app_open_acc",
    "sec_app_revol_util",
    "sec_app_open_act_il",
    "sec_app_num_rev_accts",
    "sec_app_chargeoff_within_12_mths",
    "sec_app_collections_12_mths_ex_med",
    "sec_app_mths_since_last_major_derog",
    # --- Categorical drops ---
    "emp_title",                  # 300k+ unique values
]

In [6]:
file_path = '../data/raw/loan_preprocessed.parquet'

def read_data(file_path = file_path):
    """Reads the dataset from the preprocessed data - Parquet file."""
    logger.info(f"Reading data from {file_path}...")
    try: 
        df = pd.read_parquet(file_path)
        logger.success("Data read successfully.")
        return df
    except Exception as e:
        logger.error(f"Error reading data: {e}")
        raise e 


df = read_data()
df.sample(10)
# rename target column for use of helper functions




2026-03-08 20:58:26.618 | INFO     | __main__:read_data:5 - Reading data from ../data/raw/loan_preprocessed.parquet...
2026-03-08 20:58:27.130 | SUCCESS  | __main__:read_data:8 - Data read successfully.


,loan_amnt,term,emp_title,emp_length,home_ownership,annual_inc,verification_status,loan_status,purpose,addr_state,...,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog
575244,26000,60 months,Senior Underwriter,10+ years,RENT,75550.0,Source Verified,Fully Paid,debt_consolidation,CO,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
531985,10800,36 months,head depertment,9 years,MORTGAGE,50000.0,Source Verified,Fully Paid,debt_consolidation,CT,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
473270,35000,60 months,Foreman,10+ years,MORTGAGE,100000.0,Verified,Fully Paid,home_improvement,NJ,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
115351,10000,36 months,assembler,< 1 year,OWN,32000.0,Source Verified,Fully Paid,debt_consolidation,NY,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
805344,14400,60 months,None,None,MORTGAGE,74900.0,Source Verified,Fully Paid,debt_consolidation,TN,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1036141,27975,60 months,ENGINEER,5 years,OWN,100000.0,Source Verified,Fully Paid,other,LA,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1246167,10000,36 months,Riverside County,10+ years,MORTGAGE,65000.0,Source Verified,Fully Paid,debt_consolidation,CA,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1159002,35000,60 months,TAM,10+ years,MORTGAGE,150000.0,Source Verified,Fully Paid,home_improvement,NC,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1004621,24000,60 months,Preferred Homecare,6 years,MORTGAGE,71475.0,Verified,Fully Paid,other,AZ,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1295285,8000,36 months,Auto Technician,1 year,RENT,52000.0,Not Verified,Fully Paid,debt_consolidation,PA,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
df.columns

Index(['loan_amnt', 'term', 'emp_title', 'emp_length', 'home_ownership',
       'annual_inc', 'verification_status', 'loan_status', 'purpose',
       'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line',
       'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record',
       'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
       'collections_12_mths_ex_med', 'mths_since_last_major_derog',
       'application_type', 'annual_inc_joint', 'dti_joint',
       'verification_status_joint', 'acc_now_delinq', 'tot_coll_amt',
       'tot_cur_bal', 'open_acc_6m', 'open_act_il', 'open_il_12m',
       'open_il_24m', 'mths_since_rcnt_il', 'total_bal_il', 'il_util',
       'open_rv_12m', 'open_rv_24m', 'max_bal_bc', 'all_util',
       'total_rev_hi_lim', 'inq_fi', 'total_cu_tl', 'inq_last_12m',
       'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util',
       'chargeoff_within_12_mths', 'delinq_amnt', 'mo_sin_old_il_acct',
       'mo_sin_old_rev_tl_op', 'm

In [13]:

df_new= df.drop(columns=COLS_TO_DROP, errors='ignore')
df_new.sample(10)

,loan_amnt,term,emp_length,home_ownership,annual_inc,verification_status,loan_status,purpose,addr_state,dti,...,mo_sin_old_rev_tl_op,mort_acc,num_actv_rev_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,percent_bc_gt_75,pub_rec_bankruptcies,tot_hi_cred_lim,total_bc_limit
308861,35000,36 months,None,OWN,72000.0,Verified,Charged Off,debt_consolidation,KY,12.03,...,632.0,0.0,8.0,17.0,31.0,8.0,10.0,0.0,82700.0,67300.0
398348,10000,36 months,< 1 year,MORTGAGE,47000.0,Not Verified,Fully Paid,credit_card,NM,19.41,...,198.0,4.0,6.0,6.0,18.0,3.0,75.0,0.0,175846.0,19000.0
396347,14400,60 months,8 years,RENT,60000.0,Not Verified,Fully Paid,debt_consolidation,AL,24.74,...,124.0,0.0,3.0,9.0,13.0,3.0,25.0,0.0,152811.0,12200.0
1171173,9000,36 months,5 years,OWN,40000.0,Verified,Fully Paid,debt_consolidation,FL,23.55,...,487.0,1.0,3.0,3.0,18.0,2.0,100.0,1.0,33773.0,12100.0
833580,25500,36 months,10+ years,MORTGAGE,96656.4,Verified,Fully Paid,debt_consolidation,TX,7.28,...,336.0,0.0,4.0,7.0,17.0,4.0,40.0,0.0,87712.0,51500.0
921959,2000,36 months,10+ years,RENT,80000.0,Verified,Fully Paid,debt_consolidation,NY,27.03,...,212.0,0.0,4.0,4.0,6.0,4.0,100.0,0.0,82499.0,31100.0
214553,12000,36 months,< 1 year,MORTGAGE,100000.0,Source Verified,Charged Off,debt_consolidation,TX,13.80,...,188.0,3.0,6.0,13.0,19.0,6.0,66.7,0.0,227842.0,36400.0
1125229,13000,36 months,10+ years,RENT,175000.0,Not Verified,Fully Paid,credit_card,AL,4.28,...,279.0,4.0,3.0,6.0,21.0,3.0,0.0,0.0,62770.0,24000.0
417952,12500,36 months,10+ years,MORTGAGE,38500.0,Source Verified,Charged Off,debt_consolidation,OH,24.92,...,265.0,5.0,4.0,6.0,17.0,4.0,50.0,2.0,36706.0,5300.0
151811,18000,60 months,10+ years,MORTGAGE,67500.0,Not Verified,Charged Off,debt_consolidation,TX,26.84,...,129.0,1.0,5.0,5.0,10.0,5.0,0.0,0.0,175209.0,6500.0


In [14]:
df_new.columns

Index(['loan_amnt', 'term', 'emp_length', 'home_ownership', 'annual_inc',
       'verification_status', 'loan_status', 'purpose', 'addr_state', 'dti',
       'delinq_2yrs', 'earliest_cr_line', 'inq_last_6mths', 'open_acc',
       'pub_rec', 'revol_bal', 'revol_util', 'tot_cur_bal', 'open_acc_6m',
       'open_act_il', 'open_il_12m', 'open_il_24m', 'il_util', 'open_rv_12m',
       'open_rv_24m', 'all_util', 'total_rev_hi_lim', 'inq_fi', 'inq_last_12m',
       'acc_open_past_24mths', 'avg_cur_bal', 'bc_util',
       'mo_sin_old_rev_tl_op', 'mort_acc', 'num_actv_rev_tl', 'num_op_rev_tl',
       'num_rev_accts', 'num_rev_tl_bal_gt_0', 'percent_bc_gt_75',
       'pub_rec_bankruptcies', 'tot_hi_cred_lim', 'total_bc_limit'],
      dtype='object')

In [16]:
OUTPUT_DIR = "../data/processed"
df_new.to_parquet(f"{OUTPUT_DIR}/loan_selected.parquet", index=False)
